In [1]:
import torch

In [191]:
"""
线性回归（从零实现）

    用 NumPy 或 PyTorch 实现带梯度下降的线性回归（不调用 nn.Linear）。
    要求：定义损失函数（MSE）、手动更新权重、支持多特征。
"""
import numpy as np

# 首先定义输入，(num_sample, feature_dim)
# 输出 y = XW + b
class LinearRegression:
    def __init__(self, d_model=512):
        self.W = torch.randn( (d_model, 1), dtype=torch.float32, requires_grad=True)
        self.b = torch.tensor(0., dtype=torch.float32, requires_grad=True)
        
    def forward(self, x):
        # x = torch.tensor(x, dtype=torch.float32) # 重新包装x会破坏计算图， 但是因为x是非参数，所以没影响
        return x @ self.W + self.b

def cal_loss(label, y):
    label = torch.tensor(label, dtype=torch.float32).reshape(-1, 1) # [num_sampel, 1]
    # y = y
    # 计算MSE
    # loss = torch.sqrt((label - y)**2)
    # loss = torch.sum(loss) / loss.shape[0] # 这是平均绝对误差MAE,不是MSE，不需要加开方的操作

    loss = torch.sum((label-y)**2) /y.shape[0]
    # print(loss.requires_grad) # 为什么这里又是false了
    return loss


# 构造数据
batch, feature = 32, 6
x = torch.randn(batch, feature)
true_w = torch.tensor([2.8, 1.6, -0.6, 1.5, 1.1, 0.5])
true_b = torch.tensor([1.8])
y = x @ true_w + true_b
y += torch.randn_like(y) * 0.1
# model = LinearRegression(d_model=feature)
print(x.shape, y.shape)



torch.Size([32, 6]) torch.Size([32])


In [170]:
def train(model, x, y, epochs=3000, lr=0.001):
    for epoch in range(epochs):
        # 计算模型输出
        output = model.forward(x)
  
        # 计算损失
        loss = cal_loss(y, output)
        # if epoch % 10 ==0:
        #     print(f"epoch:{epoch}, loss:{loss}")

        # 反向传播
        loss.backward()
        # 更新权重
        with torch.no_grad():
            model.W -= model.W.grad * lr # = 和 -=也有区别，一个是重新生成一个新参数， 一个是原地操作
            model.b -= model.b.grad * lr
            # 梯度清空
            model.W.grad.zero_()
            model.b.grad.zero_()
train(model, x, y)

print(f"true_w:{true_w}")
print(f"learned_w:{model.W}")
print(f"true_b:{true_b}")
print(f"learned_b:{model.b}")

C:\Users\Wtz\AppData\Local\Temp\ipykernel_31240\1032966357.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32) # 重新包装x会破坏计算图， 原来根因在这
C:\Users\Wtz\AppData\Local\Temp\ipykernel_31240\1032966357.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label = torch.tensor(label, dtype=torch.float32).reshape(-1, 1) # [num_sampel, 1]


true_w:tensor([[ 2.8000],
        [ 1.6000],
        [-0.6000],
        [ 1.5000],
        [ 1.1000],
        [ 0.5000]])
learned_w:tensor([[ 2.7553],
        [ 1.5868],
        [-0.6316],
        [ 1.4525],
        [ 1.1297],
        [ 0.5288]], requires_grad=True)
true_b:tensor([1.8000])
learned_b:1.7983088493347168


In [183]:
# 如果使用接口呢？
import torch.nn as nn
class Model(nn.Module):
    def __init__(self, in_channel=512):
        super().__init__()
        self.layer = nn.Linear(in_channel, 1)
    def forward(self, x:torch.tensor):
        assert x.dim()==2, f"need x dim 2 but got {x.dim()}"
        return self.layer(x)

def train(model, optim, x, label, epochs=5000):
    critiel = nn.MSELoss()
    for epoch in range(epochs):
        output = model(x)
        loss = critiel(output, label)
        loss.backward()
        optim.step()
        optim.zero_grad()

        if epoch % 20 == 0:
            print(f"epoch: {epoch}, loss: {loss}")

model1 = Model(in_channel=6)
optim = torch.optim.Adam(model1.parameters(), lr=0.001)
train(model1, optim, x, y)

epoch: 0, loss: 16.347379684448242
epoch: 20, loss: 15.930269241333008
epoch: 40, loss: 15.522287368774414
epoch: 60, loss: 15.124290466308594
epoch: 80, loss: 14.736353874206543
epoch: 100, loss: 14.358304023742676
epoch: 120, loss: 13.989906311035156
epoch: 140, loss: 13.630912780761719
epoch: 160, loss: 13.28108024597168
epoch: 180, loss: 12.94015884399414
epoch: 200, loss: 12.607912063598633
epoch: 220, loss: 12.284090042114258
epoch: 240, loss: 11.968453407287598
epoch: 260, loss: 11.6607666015625
epoch: 280, loss: 11.360794067382812
epoch: 300, loss: 11.068310737609863
epoch: 320, loss: 10.783090591430664
epoch: 340, loss: 10.50491714477539
epoch: 360, loss: 10.233579635620117
epoch: 380, loss: 9.968879699707031
epoch: 400, loss: 9.710617065429688
epoch: 420, loss: 9.458610534667969
epoch: 440, loss: 9.21268081665039
epoch: 460, loss: 8.972661018371582
epoch: 480, loss: 8.73839282989502
epoch: 500, loss: 8.509723663330078
epoch: 520, loss: 8.286514282226562
epoch: 540, loss: 8.06

In [ ]:
# MLP
class SimpleMLP(nn.Module):
    def __init__(self, in_channel, out_channel=10):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(in_channel, 2048),
            nn.ReLU(),
            nn.Linear(2048, out_channel)
        )
    def forward(self, x):
        return self.model(x)

def train(model, optim, x, label, epochs=5000):
    critiel = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        output = model(x) #[num_sample, 10] 十个类别， 用crossEntrophy
        print(output.shape, label.shape)
        loss = critiel(output, label)
        loss.backward()
        optim.step()
        optim.zero_grad()

        if epoch % 20 == 0:
            print(f"epoch: {epoch}, loss: {loss}")

simple_mlp = SimpleMLP(in_channel=6)
optim = torch.optim.Adam(simple_mlp.parameters(), lr=0.001)
train(simple_mlp, optim=optim, x=x, label=y, epochs=100)

torch.Size([32, 10]) torch.Size([32, 1])


RuntimeError: 0D or 1D target tensor expected, multi-target not supported

In [ ]:
# 实现一个交叉熵
label = torch.randn(32, 10) # 这里怎么生成one-hot编码
one_hot = torch.zeros_like(label)
one_hot = one_hot.scatter(1, torch.argmax(label, dim=1, keepdim=True), 1.0)

output = torch.randn(32, 10) # 对应输出的
def softmax(x, dim):
    b = torch.sum(torch.exp(-x), dim=dim, keepdim=True) # keepdim广播
    return torch.exp(-x) / b
output = softmax(output, dim=1)

# 计算交叉熵
loss = - torch.mean(torch.log(output[torch.where(one_hot>0)])) # 有正有负，不能直接Log， 或者也可用矩阵点乘
print(loss)

tensor(2.7523)


In [ ]:
# 实现一个RNN
batch, seq_len, d_model = 5, 10, 512
x = torch.rand(batch, seq_len ,d_model)

class SimpleRNN(nn.Module):
    def __init__(self,):
        super().__init__()
        self.model = nn.RNN(512, 1024, 1, batch_first=True)
        
    def forward(self, x):
        """
        input: x [batch, seq_len, d_model]
        output: [batch, 1, d_model] 新的token
        """
